# CSBI-PA — PhreshPhish import, training and results (Colab / Kaggle)

Runs the whole offline track end to end:

1. streams a balanced sample of **PhreshPhish** (Hugging Face `phreshphish/phreshphish`, CC-BY-4.0, anti-phishing research only) — no 36 GB download
2. runs every saved page through the CSBI layers (URL, HTML, layers 3–5, TBDF, kit fingerprints)
3. trains all baselines + CSBI-PA, runs the campaign experiments, builds the kit index and the report
4. saves everything you need into **one zip** (`csbi_results.zip`)

**Runtime:** CPU is enough (no GPU needed). On Colab's free CPU expect ~1–2 h for 30k pages; Kaggle (4 CPUs) is faster.
**Kaggle (recommended):** turn on *Settings → Internet*, then *Save Version → Save & Run All*. It runs in the background on Kaggle's servers for up to 12 h, so you can close the laptop.

In [ ]:
# 1. Get the code
# Kaggle keeps only files under /kaggle/working; Colab uses /content
import os
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO = f'{BASE}/CSBI'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --branch full-build https://github.com/SamyukthaGS-07/CSBI.git {REPO}
%cd {REPO}
!pip -q install -r requirements.txt datasets
print('working in', REPO)

In [ ]:
# 2. Settings — change the sample size here
N_PHISH  = 15000   # phishing pages to import
N_BENIGN = 15000   # legitimate pages to import
FREE_SHARE = 0.35  # aim for this share of free-hosted pages in each class
import os; WORKERS = os.cpu_count()
print("CPUs:", WORKERS)

In [ ]:
# 3. Import (streams from Hugging Face; resumable — re-running skips pages already imported)
!python module1_ingestion/phreshphish.py --n-phish {N_PHISH} --n-benign {N_BENIGN} --free-share {FREE_SHARE} --workers {WORKERS}

In [ ]:
# 4. Train every model + baselines (campaign-disjoint 5-fold CV + temporal split)
!python module5_decision/train.py --track offline

In [ ]:
# 5. Campaign experiments (kit clustering, label efficiency, lead time), kit index, report
!python evaluation/campaign_experiments.py
!python module4_clustering/build_index.py
!python evaluation/make_report.py

In [ ]:
# 6. Look at the headline results
from IPython.display import Markdown, display
display(Markdown(open('results/REPORT.md').read()[:6000]))

In [ ]:
# 7. Package results + trained model (+ the store, without raw HTML) into one zip
# Raw page HTML is deleted first: features and kit fingerprints are already in the store,
# and this keeps the download small.
!rm -rf data/snapshots
!zip -qr {BASE}/csbi_results.zip results module5_decision/artifacts module4_clustering/artifacts data/store/scans.db
!ls -lh {BASE}/csbi_results.zip
try:
    from google.colab import files; files.download(f'{BASE}/csbi_results.zip')
except Exception:
    print('Kaggle: when the run finishes, open this version -> Output -> download csbi_results.zip')